In [ ]:
import json, re, asyncio
from groq import Groq
from openai import AsyncOpenAI
from datasets import load_dataset

In [ ]:
from openai import OpenAI
from openai import AsyncOpenAI
API_KEY = "Your API key here"
client = AsyncOpenAI(base_url="https://openrouter.ai/api/v1", api_key=API_KEY)



In [ ]:


MODEL    = "deepseek/deepseek-v4-flash"
SAMPLES  = 5         
PER_CALL = 10           # asserts requested 

prompts = json.load(open("../data/prompts.json"))
SEM     = asyncio.Semaphore(12)

TMPL = """Write {n} assert statements testing the function below.

Rules:
- Output ONLY lines of the form: assert {ep}(...) == ...
- For functions returning floats, prefer: assert abs({ep}(...) - expected) < 1e-6
- No imports, no explanations, no comments, no test functions, no markdown
- Cover edge cases: empty input, single element, zero, negatives, boundaries

````python
{prompt}
```"""

def parse(text, ep):
    out = []
    for line in text.splitlines():
        line = line.strip().rstrip(",")
        if not line.startswith("assert ") or f"{ep}(" not in line:
            continue
        try:
            compile(line, "<t>", "exec")
        except SyntaxError:
            continue
        out.append(line)
    return out

async def one(tid, info, k):
    async with SEM:
        for attempt in range(3):
            try:
                r = await client.chat.completions.create(
    model=MODEL,
    temperature=0.8,
    max_tokens=4000,                              
    extra_body={"reasoning": {"enabled": True}}, 
    messages=[{"role": "user", "content": TMPL.format(
        n=PER_CALL, ep=info["entry_point"], prompt=info["prompt"])}])
                return tid, parse(r.choices[0].message.content, info["entry_point"])
            except TypeError:
                raise                     
            except Exception as e:
                if attempt == 2:
                    print(f"  {tid} sample {k} failed: {e}")
                    return tid, []
                await asyncio.sleep(2 ** attempt)

async def main():
    jobs = [one(t, i, k) for t, i in prompts.items() for k in range(SAMPLES)]
    results = await asyncio.gather(*jobs)

    merged = {}
    for tid, asserts in results:
        merged.setdefault(tid, [])
        merged[tid].extend(asserts)


    tests = {}
    for tid, lst in merged.items():
        seen, keep = set(), []
        for a in lst:
            key = re.sub(r"\s+", " ", a).strip()
            if key not in seen:
                seen.add(key); keep.append(a)
        tests[tid] = keep

    json.dump(tests, open("../data/reasoned_tests.json", "w"), indent=1)
    n = sorted(len(v) for v in tests.values())
    print(f"{sum(n)} tests | min {n[0]} | median {n[len(n)//2]} | max {n[-1]}")
    thin = [t for t, v in tests.items() if len(v) < 8]
    if thin: print(f"thin tasks ({len(thin)}): {thin[:10]}")


tid0 = next(iter(prompts))
_, sample = await one(tid0, prompts[tid0], 0)
print(f"smoke test ({tid0}): {len(sample)} asserts")
for a in sample[:3]: print("  ", a)
assert sample, "smoke test returned no asserts — fix before running the full matrix"

await main()
